# Skeleton 2D To Video Pixel Mapping With YOLO Hand Boxes

Use hand YOLO bounding boxes as weak visual anchors to compare ZED skeleton `u/v` hand keypoints against world-camera video pixels, then estimate an affine mapping:

`video_x = a * skeleton_u + b * skeleton_v + c`

`video_y = d * skeleton_u + e * skeleton_v + f`

In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATASET_ROOT = Path("/home/julius/datasets/seminar_object_detection")
VIDEO_ROOT = DATASET_ROOT / "iAssistADL_dataset_export_2026-05-18" / "world_camera_videos"
YOLO_HAND_ROOT = DATASET_ROOT / "lower_feature_outcome" / "yolo_hand_outcome_csv"
SKELETON_ROOT = DATASET_ROOT / "lower_feature_outcome" / "aligned_sensor_with_YOLO"
OUTPUT_ROOT = DATASET_ROOT / "video_demonstration" / "skeleton_yolo_mapping"

SELECTED_FULL_ID = "IA_4_H_02_Chaos_several_Tasks_combine4"
MIN_YOLO_CONFIDENCE = 0.25
MAX_MATCH_DISTANCE_PX = 260
RESIDUAL_KEEP_QUANTILE = 0.80
MIN_MATCHES_FOR_AFFINE = 20
RANDOM_SEED = 42
MAX_FRAMES_FOR_BODY_SELECTION = 800
MAX_FRAMES_FOR_FINAL_FIT = 2000

HAND_JOINTS = [
    "left_wrist",
    "left_hand_index_1",
    "left_hand_middle_4",
    "left_hand_pinky_1",
    "left_hand_thumb_4",
    "right_wrist",
    "right_hand_index_1",
    "right_hand_middle_4",
    "right_hand_pinky_1",
    "right_hand_thumb_4",
]

def participant_from_full_id(full_id):
    return "_".join(full_id.split("_")[:4])


def list_available_full_ids(participant_id=None):
    available = []
    for csv_path in sorted(YOLO_HAND_ROOT.glob("*_world_camera.csv")):
        full_id = csv_path.name.replace("_world_camera.csv", "")
        current_participant = participant_from_full_id(full_id)
        if participant_id is not None and current_participant != participant_id:
            continue
        current_video = VIDEO_ROOT / current_participant / f"{full_id}_world_camera.mp4"
        current_skeleton = SKELETON_ROOT / current_participant / full_id
        if current_video.exists() and current_skeleton.exists():
            available.append(full_id)
    return available


participant_id = participant_from_full_id(SELECTED_FULL_ID)
video_path = VIDEO_ROOT / participant_id / f"{SELECTED_FULL_ID}_world_camera.mp4"
yolo_path = YOLO_HAND_ROOT / f"{SELECTED_FULL_ID}_world_camera.csv"
skeleton_dir = SKELETON_ROOT / participant_id / SELECTED_FULL_ID

missing_paths = [path for path in [video_path, yolo_path, skeleton_dir] if not path.exists()]
if missing_paths:
    print("SELECTED_FULL_ID not found with complete video / YOLO / skeleton inputs:")
    for path in missing_paths:
        print(f"  missing: {path}")
    available_ids = list_available_full_ids(participant_id)
    print(f"\nAvailable full_id values for {participant_id}:")
    for full_id in available_ids:
        print(f"  {full_id}")
    raise FileNotFoundError(missing_paths[0])

cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {video_path}")
VIDEO_WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
VIDEO_HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
VIDEO_FPS = cap.get(cv2.CAP_PROP_FPS) or 30.0
VIDEO_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

{
    "selected_full_id": SELECTED_FULL_ID,
    "video": str(video_path),
    "yolo": str(yolo_path),
    "skeleton_dir": str(skeleton_dir),
    "video_size": (VIDEO_WIDTH, VIDEO_HEIGHT),
    "video_frames": VIDEO_FRAMES,
    "fps": VIDEO_FPS,
}

## Load YOLO Hand Boxes

In [ ]:
def load_yolo_boxes(path, min_confidence=0.25):
    df = pd.read_csv(path)
    for col in ["frame_id", "confidence", "x1", "y1", "x2", "y2"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["frame_id", "confidence", "x1", "y1", "x2", "y2"]).copy()
    df = df[df["confidence"] >= min_confidence].copy()
    df["frame_id"] = df["frame_id"].astype(int)
    df["bbox_cx"] = (df["x1"] + df["x2"]) / 2.0
    df["bbox_cy"] = (df["y1"] + df["y2"]) / 2.0
    df["bbox_w"] = df["x2"] - df["x1"]
    df["bbox_h"] = df["y2"] - df["y1"]
    return df.sort_values(["frame_id", "confidence"], ascending=[True, False]).reset_index(drop=True)


yolo_df = load_yolo_boxes(yolo_path, MIN_YOLO_CONFIDENCE)
yolo_by_frame = {frame_id: frame_df for frame_id, frame_df in yolo_df.groupby("frame_id", sort=False)}

display(yolo_df.head())
{
    "detections": len(yolo_df),
    "frames_with_detections": yolo_df["frame_id"].nunique(),
    "mean_confidence": float(yolo_df["confidence"].mean()) if len(yolo_df) else 0,
}

## Load Skeleton Hand Points

In [ ]:
def skeleton_usecols(columns):
    wanted = {"video_frame_idx"}
    for joint in HAND_JOINTS:
        wanted.add(f"{joint}_u")
        wanted.add(f"{joint}_v")
    return [col for col in columns if col in wanted]


def load_skeleton_points(path):
    header = pd.read_csv(path, nrows=0).columns.tolist()
    usecols = skeleton_usecols(header)
    if "video_frame_idx" not in usecols:
        raise ValueError(f"Missing video_frame_idx in {path}")
    df = pd.read_csv(path, usecols=usecols)
    df["video_frame_idx"] = pd.to_numeric(df["video_frame_idx"], errors="coerce")
    df = df.dropna(subset=["video_frame_idx"]).copy()
    df["video_frame_idx"] = df["video_frame_idx"].round().astype(int)

    rows = []
    body_id = path.stem.replace("zed__skeleton__Body_", "")
    for joint in HAND_JOINTS:
        u_col = f"{joint}_u"
        v_col = f"{joint}_v"
        if u_col not in df.columns or v_col not in df.columns:
            continue
        part = df[["video_frame_idx", u_col, v_col]].copy()
        part = part.rename(columns={"video_frame_idx": "frame_id", u_col: "u", v_col: "v"})
        part["u"] = pd.to_numeric(part["u"], errors="coerce")
        part["v"] = pd.to_numeric(part["v"], errors="coerce")
        part = part.dropna(subset=["u", "v"])
        part = part[np.isfinite(part["u"]) & np.isfinite(part["v"])]
        part = part[(part["u"] >= 0) & (part["v"] >= 0)].copy()
        part["joint"] = joint
        part["side"] = "left" if joint.startswith("left_") else "right"
        part["body_id"] = body_id
        rows.append(part)

    if not rows:
        return pd.DataFrame(columns=["frame_id", "u", "v", "joint", "side", "body_id"])
    out = pd.concat(rows, ignore_index=True)
    out = out[out["frame_id"].isin(yolo_by_frame.keys())].copy()
    out = out.groupby(["frame_id", "joint", "side", "body_id"], as_index=False)[["u", "v"]].median()
    return out


skeleton_paths = sorted(skeleton_dir.glob("zed__skeleton__Body_*.csv"))
print(f"skeleton body tracks: {len(skeleton_paths)}")
skeleton_paths[:5]

## Estimate Skeleton `u/v` To Video Pixel Mapping

In [ ]:
def initial_affine_from_range(points_df):
    uv = points_df[["u", "v"]].dropna()
    if uv.empty:
        return np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
    u_max = uv["u"].quantile(0.99)
    v_max = uv["v"].quantile(0.99)
    if u_max <= 2.0 and v_max <= 2.0:
        return np.array([[VIDEO_WIDTH, 0.0, 0.0], [0.0, VIDEO_HEIGHT, 0.0]])
    return np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])


def apply_affine(uv, affine):
    uv = np.asarray(uv, dtype=float)
    ones = np.ones((len(uv), 1), dtype=float)
    return np.column_stack([uv, ones]) @ affine.T


def fit_affine(src_uv, dst_xy):
    src_uv = np.asarray(src_uv, dtype=float)
    dst_xy = np.asarray(dst_xy, dtype=float)
    X = np.column_stack([src_uv, np.ones(len(src_uv))])
    coef_x, *_ = np.linalg.lstsq(X, dst_xy[:, 0], rcond=None)
    coef_y, *_ = np.linalg.lstsq(X, dst_xy[:, 1], rcond=None)
    return np.vstack([coef_x, coef_y])


def sample_frame_ids(frame_ids, max_frames=None, seed=RANDOM_SEED):
    frame_ids = np.array(sorted({int(frame_id) for frame_id in frame_ids}), dtype=int)
    if max_frames is None or len(frame_ids) <= max_frames:
        return set(frame_ids.tolist())
    rng = np.random.default_rng(seed)
    sampled = rng.choice(frame_ids, size=max_frames, replace=False)
    return set(int(frame_id) for frame_id in sampled)


def build_matches(points_df, affine, max_distance_px=260, frame_ids=None):
    if frame_ids is not None:
        points_df = points_df[points_df["frame_id"].isin(frame_ids)]
    matches = []
    for frame_id, frame_points in points_df.groupby("frame_id", sort=False):
        boxes = yolo_by_frame.get(int(frame_id))
        if boxes is None or boxes.empty or frame_points.empty:
            continue
        points = frame_points.reset_index(drop=True).copy()
        mapped = apply_affine(points[["u", "v"]].to_numpy(), affine)
        used_points = set()
        for _, box in boxes.sort_values("confidence", ascending=False).head(4).iterrows():
            center = np.array([box["bbox_cx"], box["bbox_cy"]], dtype=float)
            distances = np.linalg.norm(mapped - center, axis=1)
            order = np.argsort(distances)
            chosen = None
            for idx in order:
                if idx not in used_points:
                    chosen = int(idx)
                    break
            if chosen is None or distances[chosen] > max_distance_px:
                continue
            used_points.add(chosen)
            pt = points.iloc[chosen]
            matches.append(
                {
                    "frame_id": int(frame_id),
                    "body_id": pt["body_id"],
                    "joint": pt["joint"],
                    "side": pt["side"],
                    "u": float(pt["u"]),
                    "v": float(pt["v"]),
                    "target_x": float(center[0]),
                    "target_y": float(center[1]),
                    "distance_px": float(distances[chosen]),
                    "bbox_confidence": float(box["confidence"]),
                    "bbox_w": float(box["bbox_w"]),
                    "bbox_h": float(box["bbox_h"]),
                }
            )
    return pd.DataFrame(matches)


def estimate_affine(points_df, iterations=4, max_frames=None):
    frame_ids = sample_frame_ids(points_df["frame_id"].unique(), max_frames=max_frames)
    affine = initial_affine_from_range(points_df)
    matches = pd.DataFrame()
    for _ in range(iterations):
        matches = build_matches(points_df, affine, MAX_MATCH_DISTANCE_PX, frame_ids=frame_ids)
        if len(matches) < MIN_MATCHES_FOR_AFFINE:
            break
        residual_cutoff = matches["distance_px"].quantile(RESIDUAL_KEEP_QUANTILE)
        fit_df = matches[matches["distance_px"] <= residual_cutoff].copy()
        if len(fit_df) < MIN_MATCHES_FOR_AFFINE:
            fit_df = matches
        affine = fit_affine(fit_df[["u", "v"]].to_numpy(), fit_df[["target_x", "target_y"]].to_numpy())
    matches = build_matches(points_df, affine, MAX_MATCH_DISTANCE_PX, frame_ids=frame_ids)
    if not matches.empty:
        mapped = apply_affine(matches[["u", "v"]].to_numpy(), affine)
        matches["mapped_x"] = mapped[:, 0]
        matches["mapped_y"] = mapped[:, 1]
        matches["residual_px"] = np.linalg.norm(mapped - matches[["target_x", "target_y"]].to_numpy(), axis=1)
    return affine, matches


body_rows = []
body_results = {}
for skeleton_path in skeleton_paths:
    points_df = load_skeleton_points(skeleton_path)
    if points_df.empty:
        continue
    affine, matches = estimate_affine(points_df, max_frames=MAX_FRAMES_FOR_BODY_SELECTION)
    body_id = skeleton_path.stem.replace("zed__skeleton__Body_", "")
    body_results[body_id] = {"path": skeleton_path, "points": points_df, "affine": affine, "matches": matches}
    body_rows.append(
        {
            "body_id": body_id,
            "points": len(points_df),
            "matched_pairs": len(matches),
            "median_residual_px": float(matches["residual_px"].median()) if "residual_px" in matches and len(matches) else np.nan,
            "mean_residual_px": float(matches["residual_px"].mean()) if "residual_px" in matches and len(matches) else np.nan,
            "frames_matched": matches["frame_id"].nunique() if len(matches) else 0,
            "skeleton_path": str(skeleton_path),
        }
    )

body_score_df = pd.DataFrame(body_rows).sort_values(
    ["matched_pairs", "median_residual_px"], ascending=[False, True]
).reset_index(drop=True)
display(body_score_df.head(20))

if body_score_df.empty:
    raise RuntimeError("No skeleton body track could be matched to YOLO hand boxes.")

BEST_BODY_ID = str(body_score_df.iloc[0]["body_id"])
best = body_results[BEST_BODY_ID]
best["affine"], best["matches"] = estimate_affine(best["points"], max_frames=MAX_FRAMES_FOR_FINAL_FIT)
affine = best["affine"]
matches_df = best["matches"]
points_df = best["points"]

print(f"Selected Body ID: {BEST_BODY_ID}")
print("Affine skeleton_uv -> video_pixel:")
print(affine)
print(f"video_x = {affine[0,0]:.6f} * u + {affine[0,1]:.6f} * v + {affine[0,2]:.6f}")
print(f"video_y = {affine[1,0]:.6f} * u + {affine[1,1]:.6f} * v + {affine[1,2]:.6f}")

def map_skeleton_uv_to_video_pixel(u, v):
    mapped = apply_affine(np.array([[u, v]], dtype=float), affine)
    return float(mapped[0, 0]), float(mapped[0, 1])

display(matches_df.head())
matches_df[["residual_px", "distance_px", "bbox_confidence"]].describe()

## Visual Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(matches_df["target_x"], matches_df["mapped_x"], s=8, alpha=0.5)
axes[0].plot([0, VIDEO_WIDTH], [0, VIDEO_WIDTH], color="black", lw=1)
axes[0].set_title("x: YOLO center vs mapped skeleton")
axes[0].set_xlabel("YOLO bbox center x")
axes[0].set_ylabel("mapped skeleton x")

axes[1].scatter(matches_df["target_y"], matches_df["mapped_y"], s=8, alpha=0.5)
axes[1].plot([0, VIDEO_HEIGHT], [0, VIDEO_HEIGHT], color="black", lw=1)
axes[1].set_title("y: YOLO center vs mapped skeleton")
axes[1].set_xlabel("YOLO bbox center y")
axes[1].set_ylabel("mapped skeleton y")

axes[2].hist(matches_df["residual_px"], bins=40, color="#4c78a8")
axes[2].set_title("Mapping residual")
axes[2].set_xlabel("pixels")
axes[2].set_ylabel("matched pairs")

plt.tight_layout()
plt.show()

In [ ]:
def draw_overlay_frame(frame_id):
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_id))
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f"Could not read frame {frame_id}")

    boxes = yolo_by_frame.get(int(frame_id), pd.DataFrame())
    for _, box in boxes.iterrows():
        x1, y1, x2, y2 = [int(round(box[col])) for col in ["x1", "y1", "x2", "y2"]]
        cv2.rectangle(frame, (x1, y1), (x2, y2), (40, 220, 255), 3)
        cv2.putText(frame, f"hand {box['confidence']:.2f}", (x1, max(24, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (40, 220, 255), 2)

    frame_points = points_df[points_df["frame_id"] == int(frame_id)].copy()
    if not frame_points.empty:
        mapped = apply_affine(frame_points[["u", "v"]].to_numpy(), affine)
        for (_, pt), (x, y) in zip(frame_points.iterrows(), mapped):
            color = (80, 255, 80) if pt["side"] == "left" else (255, 120, 80)
            cv2.circle(frame, (int(round(x)), int(round(y))), 7, color, -1)
            cv2.putText(frame, pt["joint"].replace("_hand_", "_h_"), (int(round(x)) + 8, int(round(y)) - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 8))
    plt.imshow(frame_rgb)
    plt.title(f"{SELECTED_FULL_ID} frame {frame_id} | Body {BEST_BODY_ID}")
    plt.axis("off")
    plt.show()


if len(matches_df):
    preview_frame = int(matches_df.sort_values("residual_px").iloc[len(matches_df) // 2]["frame_id"])
else:
    preview_frame = int(yolo_df.iloc[0]["frame_id"])

draw_overlay_frame(preview_frame)

## Save Mapping And Matched Pairs

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
mapping_json = OUTPUT_ROOT / f"{SELECTED_FULL_ID}_body_{BEST_BODY_ID}_skeleton_to_video_pixel_affine.json"
matches_csv = OUTPUT_ROOT / f"{SELECTED_FULL_ID}_body_{BEST_BODY_ID}_skeleton_yolo_matches.csv"

mapping_payload = {
    "full_id": SELECTED_FULL_ID,
    "body_id": BEST_BODY_ID,
    "video_path": str(video_path),
    "yolo_path": str(yolo_path),
    "skeleton_path": str(best["path"]),
    "video_width": VIDEO_WIDTH,
    "video_height": VIDEO_HEIGHT,
    "affine_2x3": affine.tolist(),
    "equations": {
        "video_x": f"{affine[0,0]:.9f} * skeleton_u + {affine[0,1]:.9f} * skeleton_v + {affine[0,2]:.9f}",
        "video_y": f"{affine[1,0]:.9f} * skeleton_u + {affine[1,1]:.9f} * skeleton_v + {affine[1,2]:.9f}",
    },
    "matched_pairs": int(len(matches_df)),
    "median_residual_px": float(matches_df["residual_px"].median()) if len(matches_df) else None,
    "mean_residual_px": float(matches_df["residual_px"].mean()) if len(matches_df) else None,
}

import json
with open(mapping_json, "w") as f:
    json.dump(mapping_payload, f, indent=2)
matches_df.to_csv(matches_csv, index=False)

mapping_json, matches_csv